# 08 In-Place TTT: A Model That Updates Its Weights While Reading

If you want to serve a Test-Time Training (TTT) model, you immediately run into a structural difference: it stores "memory" in a different place than a standard inference model.

A standard model keeps its weights frozen during inference and stores all context memory in the KV cache. TTT's idea is to treat part of the model's weights as "fast weights" that are continuously updated while reading inputs, memorizing context directly in the weights. ByteDance Seed's In-Place TTT (*In-Place Test-Time Training*, arXiv 2604.06169) is called "In-Place" because it adds no new layers—instead, it updates the existing MLP down-projection matrices directly as fast weights.

Putting such a model into an inference server raises three concrete challenges:

1. **Each request needs its own isolated fast weights**: in the 4B checkpoint we use below, fast weights span 6 matrices totaling 149,422,080 parameters—about 570 MiB per copy in FP32. Concurrent requests must never overwrite each other's weights.
2. **Updates happen mid-forward**: a long prompt triggers fast-weight updates partway through prefill; during token-by-token decode, as soon as a chunk fills up, an update must also trigger immediately.
3. **The official inference code supports only a single request**: built on Hugging Face `generate`, the official implementation runs one request at a time (`assert len(input_ids) == 1, "only support bs=1 for TTT inference"` in the source code; the authors note in issue #4 that this is an implementation limitation).

In this example, we walk through four steps:

1. Explain the In-Place TTT update rule in plain terms;
2. Load a community checkpoint with pLLM's built-in `InPlaceTTTWorkload`;
3. Decode across a chunk boundary to observe fast weights updating mid-generation and the very next token reading the new weights;
4. Benchmark throughput against the official implementation on the same RTX 4090, and discuss the trade-off between FP32 and BF16 precision.

## Method and Checkpoint Source

Before running code, let's clarify two things: which weights the model updates and how, and where our checkpoint comes from.

### Method

- **Which weights are fast weights**: in selected MLP layers, the down-projection matrix $W$ (shape `[d_model, d_ff] = [2560, 9728]` in this checkpoint) serves as fast weights. This 36-layer checkpoint selects layers `0, 6, 12, 18, 24, 30` (every 6th layer). Attention layers and the other two MLP matrices remain frozen during inference.
- **How chunked updates work**: input tokens are grouped into fixed-size chunks (`1024` tokens per chunk in this checkpoint). Whenever a full chunk is read, a weight update is computed:
  $$\Delta W = \eta \cdot V^\top Z$$
  where $Z$ holds each token's intermediate MLP activation (the input right before the down projection) across the chunk; $V$ is the "target", computed from the MLP input via a learned 1D convolution (kernel size 5, applied strictly within the current chunk) followed by a learned square projection; and $\eta$ is the learning rate (`3` in this checkpoint). The fast weights then update as $W \leftarrow W + \Delta W$.
- **Who sees the updated weights**: strict chunk-level isolation—tokens in chunk $i$ only read the updates accumulated from chunks $1 \dots i-1$. If a prompt is long, several updates fire during a single prefill; any trailing tokens at the end of the prompt that do not fill a chunk are buffered and continue accumulating during decode. Once a full chunk completes, an update triggers: the final token that completes the chunk still reads the old weights, and the very next token reads the new weights.
- **No cross-request interference**: every request starts from the checkpoint's base $W$ and maintains its own independent updates.
- **No external labels needed at inference**: the 1D convolution and square projection parameters above are learned during "outer-loop" continued pretraining (the paper trains on Qwen3-4B for ~20B tokens at 32K length, then ~15B tokens at 128K length). At inference time, the model simply updates $W$ via the formula above as it reads tokens, with no external labels.
- **Two differences between the paper and the official code**: first, the paper describes targets $V$ coming from token embeddings, whereas the official code default (and this checkpoint's config) uses the MLP input (`ttt_target="hidden_states"`); second, the paper's Qwen3-4B evaluation clips each update's norm to `1e-5`, whereas the official code applies no clipping. pLLM's `InPlaceTTTWorkload` follows the official code and checkpoint config: MLP inputs as targets, without clipping.

### Checkpoint Source

The paper authors have not released model weights (in issue #1 of the official repository, they state that internal data policies prevent releasing training data and weights).

This example uses the community-uploaded checkpoint `zhongweixie/inplace-ttt-qwen3-4b-32k-ttt` (pinned to a specific revision in the code below). Its model card states it was trained from Qwen3-4B for 6000 steps at 32K context length. We verified its config and tensor shapes (6 fast-weight layers, each with a conv and projection, learning rate $\eta=3$, chunk size 1024), though we did not audit its training run and do not evaluate generation quality here—we use it solely to inspect runtime mechanics and measure throughput.

First, let's read this checkpoint's configuration to see which layers hold fast weights along with its chunk size and learning rate.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors and suppress startup logs

import argparse
from dataclasses import replace
from pathlib import Path

import torch
from vllm import SamplingParams

import pllm
from pllm.benchmark import throughput
from pllm.workloads.inplace_ttt import InPlaceTTTConfig, InPlaceTTTWorkload

CHECKPOINT = "zhongweixie/inplace-ttt-qwen3-4b-32k-ttt"  # Community-trained checkpoint, not released by the paper authors
REVISION = "1f6aa7ea4bac2149358c86afea614f9edbd73c54"
ttt = InPlaceTTTConfig.from_checkpoint(CHECKPOINT, revision=REVISION)
print("快权重所在的层:", [target.layer for target in ttt.targets])
print("块长:", ttt.chunk_size, " 学习率:", ttt.learning_rate)

快权重所在的层: [0, 6, 12, 18, 24, 30]
块长: 1024  学习率: 3.0


In the output above, `InPlaceTTTConfig.from_checkpoint` reads the fast-weight layers `[0, 6, 12, 18, 24, 30]`, chunk size `1024`, and learning rate `3.0` directly from the checkpoint's `config.json`.

## What the Workload Looks Like and How Much VRAM It Uses

Next we answer two questions: how much code does an In-Place TTT workload take in pLLM, and before loading ~8 GB of model weights onto the GPU, how can we tell upfront how much VRAM those fast weights will consume?

`InPlaceTTTWorkload` is implemented in `pllm/workloads/inplace_ttt.py` in roughly 60 lines of body code. Its core is a single method decorated with `@pllm.train_in_forward`: given one chunk of MLP inputs (`targets`), intermediate MLP activations (`inputs`), and the accumulated fast-weight delta (`delta`), it computes $V$ via conv1d + projection, computes the chunk's weight update, and returns the updated `delta` along with the output correction for that chunk. Here is a simplified excerpt:

```python
@pllm.train_in_forward(
    visibility="exact",
    state="dense_fast_weights",
    targets=targets,                                 # def targets(self): return self.config.targets
    weight="delta",
    chunk=chunk_size,                                # def chunk_size(self): return self.config.chunk_size
    tokens="all",
)
def step(self, delta=pllm.parameter("dense_fast_weights", "delta"), inputs=pllm.operand("input"),
         targets=pllm.operand("target"), conv=pllm.parameter("target_parameters", "conv"),
         proj=pllm.parameter("target_parameters", "proj")):
    values = conv1d(targets, conv) @ proj            # V: intra-chunk conv1d followed by projection
    update = (values.T @ inputs) * learning_rate     # ΔW = η · Vᵀ Z
    return delta + update, inputs @ delta.T          # new delta; correction read by this chunk
```

The workload author writes only the tensor math above—zero lines of page-table management, continuous-batching plumbing, or custom CUDA kernels. From those ~25 lines of pure math, pLLM **automatically** delivers five capabilities that the official repository (`ByteDance-Seed/In-Place-TTT`) cannot:

1. **Breaking the `bs=1` wall with continuous batching and per-request isolation**: the official inference code mutates a single cloned `down_proj` in place and hardcodes `assert len(input_ids) == 1`; pLLM automatically lowers `RequestState` onto a paged state pool so concurrent requests of different lengths and chunk frontiers read and update their own `delta` pages in the same forward pass, reclaiming pages automatically upon completion or preemption.
2. **Arbitrarily sliced Chunked Prefill & exact cross-phase chunk boundaries**: the official loop requires feeding the entire prompt at once; pLLM automatically buffers partial-chunk tail rows across steps so that no matter how vLLM slices a long prompt—or whether the 1024th token lands mid-prefill or mid-decode—the update fires on the exact chunk boundary and subsequent tokens immediately read the updated weights (`visibility="exact"`).
3. **Full-model decode CUDA Graph capture with device-side conditional updates (zero host syncs)**: hand-written chunk-boundary branches easily trigger CPU–GPU syncs or break graph capture; pLLM captures steady-state decode inside full-model CUDA graphs and wraps the 1-in-1024 chunk update in a device-side conditional node with 0 host syncs per step.
4. **Zero-copy page sharing & paged streaming readouts on non-boundary steps**: 6 layers of FP32 fast weights weigh ~570 MiB per request, so naive per-token copy-on-write and gathering would waste nearly 3 GB of HBM traffic per step; pLLM statically analyzes `step`'s FX graph to share unchanged `delta` pages read-only across the 1023/1024 non-boundary decode steps (0 bytes copied), borrows read-only `ModelCheckpoint` conv/proj weights in place, and lowers `inputs @ delta.to(inputs.dtype).T` into a paged streaming kernel that skips idle padding lanes.
5. **Single frozen BF16 base + FP32 delta accumulation**: all requests share a single read-only BF16 base model (while vLLM automatically skips `ttt_conv` / `ttt_proj` claimed by `ModelCheckpoint`) and accumulate `delta` in FP32, avoiding both base-weight duplication and BF16 mantissa swamping of small updates.

Before launching the engine to load ~8 GB of weights, let's call `pllm.explain` (see [Seeing Costs While Writing Code](../guides/explain.md)) to inspect the VRAM budget and chosen execution mode at concurrency 1 (`max_num_seqs=1`).

In [2]:
small = pllm.ModelConfig(CHECKPOINT, revision=REVISION, max_model_len=2048, max_num_seqs=1,
                         max_num_batched_tokens=2048, gpu_memory_utilization=0.4)
plan = pllm.explain(InPlaceTTTWorkload(ttt), small)
print("eager 原因:", plan.eager_reason)
for state in plan.states:
    print(f"{state.name:20s} {state.scope:8s} 每份 {state.bytes_per_root / 2**20:6.1f} MiB × {state.roots} 份")

eager 原因: 
dense_fast_weights   request  每份  714.0 MiB × 4 份
target_parameters    instance 每份   75.1 MiB × 1 份


Let's read the `pllm.explain` output line by line:

- **`eager_reason` is empty**: under default settings, exact chunk-boundary updates mid-decode run inside CUDA graphs; all subsequent runs and throughput numbers in this notebook have CUDA graphs enabled. Exiting an engine context releases its model, request state, and graphs, allowing another engine to be created in the same process.
- **`dense_fast_weights` (`714.0 MiB × 4` copies)**: this is per-request exclusive state (`request` scope). Each `714.0 MiB` slot holds the 6 layers of FP32 `delta` matrices (~570 MiB) plus the buffer for partial-chunk input rows. Four copies are reserved for 1 request so the engine can hold both the newly written version and the version currently being read across concurrent/in-flight steps.
- **`target_parameters` (`75.1 MiB × 1` copy)**: these are the 6 layers of 1D conv and square projection weights, stored in `instance` scope and shared across all requests at ~75 MiB total.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, revision=REVISION)
text = Path(argparse.__file__).read_text()  # A long text available on every machine: source code of the argparse module
ids = tokenizer(text, add_special_tokens=False).input_ids
print("长文本共", len(ids), "个 token")

prompt = [{"prompt_token_ids": ids[:1023]}]  # One token short of a full chunk (1024)
five = SamplingParams(temperature=0.0, max_tokens=5, ignore_eos=True, logprobs=0)
runs = {}
for name, config in (("TTT", ttt), ("不更新", replace(ttt, chunk_size=2048))):
    with pllm.make_llm(small, workload=InPlaceTTTWorkload(config)) as llm:
        sample = llm.generate(prompt, five, use_tqdm=False)[0].outputs[0]
        runs[name] = [(token, step[token].logprob) for token, step in zip(sample.token_ids, sample.logprobs)]
        print(f"{name}: 快权重更新了 {llm.diagnostics().update_count} 次")

print("两组生成的 token 相同:", [t for t, _ in runs["TTT"]] == [t for t, _ in runs["不更新"]])
print("\n生成的 token   算出它的位置   TTT 的 logprob   不更新的 logprob   差")
for k, ((token, a), (_, b)) in enumerate(zip(runs["TTT"], runs["不更新"])):
    print(f"{tokenizer.decode(token)!r:14s} {1022 + k:6d}      {a:10.4f}       {b:10.4f}      {a - b:+.4f}")

长文本共 19736 个 token


TTT: 快权重更新了 6 次


不更新: 快权重更新了 0 次


两组生成的 token 相同: True

生成的 token   算出它的位置   TTT 的 logprob   不更新的 logprob   差
' _'             1022         -0.4400          -0.4400      +0.0000
'get'            1023         -1.9058          -1.9058      +0.0000
'_default'       1024         -2.2565          -2.3191      +0.0626
'_action'        1025         -1.8538          -1.8214      -0.0324
'(action'        1026         -1.7855          -1.7124      -0.0732


The output makes the chunk-boundary update timing unmistakable:

- **Update counts**: the `TTT` run performed **6 updates**, corresponding to 6 fast-weight layers × 1 completed chunk; the control run (updates disabled by setting `chunk_size=2048`, labeled `不更新` in the output) performed **0 updates**.
- **First two generated tokens (logprob diff `+0.0000`)**:
  - The 1st generated token `' _'` is computed from position `1022` (the last token of the prompt, i.e., the 1023rd token). The 1024-token chunk is not full yet, so both runs report the exact same logprob `-0.4400`.
  - The 2nd generated token `'get'` is computed from position `1023` (the 1024th token overall). Arriving at position `1023` completes the first 1024-token chunk and triggers the fast-weight update, **yet the final token that completes the chunk still reads the old weights**, so both logprobs remain identical (`-1.9058`, diff `+0.0000`).
- **From the 3rd generated token onward (logprobs diverge)**:
  - The 3rd token `'_default'` is computed from position `1024` (the 1st position of the second chunk). It reads the newly updated fast weights, shifting its logprob from `-2.3191` (without updates) to `-2.2565` (diff `+0.0626`).
  - Positions `1025` and `1026` likewise read the updated weights, with logprob diffs of `-0.0324` and `-0.0732`. (Across these 5 greedy decode steps, the argmax tokens happen to match despite the logprob shifts: `True` in the output.)

This is the exact chunk-level visibility defined in the paper. It happens automatically during vLLM's token-by-token decode without any hand-written scheduling code in `InPlaceTTTWorkload` to buffer chunks or flip weight visibility.

Having verified correctness, let's check the next natural question: how fast does this run in pLLM compared to the official Hugging Face implementation?

In [4]:
def measure(llm, length: int, batch: int) -> float:
    """Throughput of batch identical prompts (first length tokens), each generating 256 greedy tokens."""
    prompts = [{"prompt_token_ids": ids[:length]}] * batch
    return throughput(llm, prompts, SamplingParams(temperature=0.0, max_tokens=256, ignore_eos=True))


long = pllm.ModelConfig(CHECKPOINT, revision=REVISION, max_model_len=16640, max_num_seqs=1,
                        max_num_batched_tokens=8192, enable_chunked_prefill=True, gpu_memory_utilization=0.5)
for dtype in (torch.float32, torch.bfloat16):
    with pllm.make_llm(long, workload=InPlaceTTTWorkload(ttt, dtype=dtype)) as llm:
        for length in (4096, 16384):
            print(f"pllm {str(dtype)[6:]:8s} 1 个请求  prompt {length:5d}: {measure(llm, length, 1):6.2f} token/s")

pllm float32  1 个请求  prompt  4096:  73.02 token/s


pllm float32  1 个请求  prompt 16384:  47.97 token/s


pllm bfloat16 1 个请求  prompt  4096:  61.02 token/s


pllm bfloat16 1 个请求  prompt 16384:  42.32 token/s


In [5]:
# 4 requests with one fast-weight state each (10 copies total including in-flight versions), plus CUDA graph private pools; leave slightly less VRAM for vLLM on a 24 GiB GPU.
four = replace(long, max_model_len=4416, max_num_seqs=4, max_num_batched_tokens=4416, gpu_memory_utilization=0.48)
with pllm.make_llm(four, workload=InPlaceTTTWorkload(ttt)) as llm:
    print(f"pllm float32  4 个请求  prompt  4096: {measure(llm, 4096, 4):6.2f} token/s")

pllm float32  4 个请求  prompt  4096: 171.56 token/s


## Throughput Comparison with the Official Implementation

In this step, we answer: on the same hardware and identical input/output lengths, how does pLLM's throughput compare to the official implementation, and how much does throughput scale when moving from 1 request to concurrent requests?

First, let's align the benchmarking protocol on both sides:

- **Hardware and input**: the same RTX 4090 GPU, using the exact same prompt tokens (the first 4096 or 16384 tokens of `argparse` above).
- **Decoding settings**: greedy decoding, `ignore_eos=True` (no early stop on EOS), generating exactly 256 tokens.
- **Timing method**: 1 warmup round followed by 3 timed rounds taking the median; throughput is `total generated tokens / wall-clock time of generate (including prefill)`. That is exactly what `pllm.benchmark.throughput` measures.

The official implementation comes from `ByteDance-Seed/In-Place-TTT` using Hugging Face `generate`, BF16 precision, and FlashAttention 2. Because it depends on a separate Python environment (`torch 2.8.0`, `transformers 4.57.3`), we ran it separately on the same machine and same RTX 4090:

Official benchmark snippet (run in its own environment):

```python
from inference_model.hf_qwen3.modeling_qwen3 import Qwen3ForCausalLM  # Model class from the official repository

model = Qwen3ForCausalLM.from_pretrained(checkpoint, dtype=torch.bfloat16,
                                         attn_implementation="flash_attention_2").eval().cuda()
model.generate(input_ids=prompt, attention_mask=torch.ones_like(prompt), max_new_tokens=256,
               min_new_tokens=256, do_sample=False, eos_token_id=None, pad_token_id=151643, use_cache=True)
```

Putting the numbers from the two cells above side by side with the official implementation (all in `token/s`; pLLM on vLLM 0.32.0 with CUDA graphs enabled; official measured on the same day, same GPU, same protocol):

| | 1 request, prompt 4096 | 1 request, prompt 16384 | 4 requests, prompt 4096 |
| --- | --- | --- | --- |
| Official HF implementation (BF16) | 54.79 | 34.89 | Unsupported (hardcoded `bs=1`) |
| pLLM, `float32` fast weights | 73.02 | 47.97 | 171.56 (4 requests total) |
| pLLM, `bfloat16` fast weights | 61.02 | 42.32 | Not measured |

Several conclusions jump straight out of the table:

- **Single-request comparison**: pLLM's default `float32` mode is ~33% faster than the official code at prompt 4096 (`73.02` vs. `54.79`) and ~37% faster at prompt 16384 (`47.97` vs. `34.89`); `bfloat16` mode (reproducing the official BF16 rounding behavior) is also ~11% (`61.02`) and ~21% (`42.32`) faster. Why is `float32` substantially faster than `bfloat16` here? Because in `float32` mode pLLM detects that `inputs @ delta.to(inputs.dtype).T` is a pure linear readout and streams the residual correction directly from shared read-only FP32 pages on non-boundary steps, whereas `bfloat16` mode combines and rounds base weights with `delta` into BF16 and runs two matmuls to emulate in-place BF16 rounding bit-for-bit.
- **Multi-request concurrency**: with 4 concurrent requests, pLLM (`float32` fast weights) reaches `171.56 token/s` aggregate—**`3.13×`** the official single-request implementation (`54.79 token/s`) and **`2.35×`** single-request pLLM (`73.02 token/s`)—whereas the official implementation's `bs=1` limitation cannot serve concurrent requests. Four requests hold 8 fast-weight versions total (714 MiB each) plus CUDA graph private pools, fitting on a 24 GiB card at `gpu_memory_utilization=0.48`.
- **Bit-for-bit output comparison**: pLLM's generated token sequence is not bit-for-bit identical to the official implementation. Matmul tiling and floating-point reduction order differ between the two stacks, and whenever greedy decoding encounters two candidate tokens with nearly tied scores, tiny numerical differences can steer subsequent steps down a different token path.

## Precision Choice: FP32 or BF16 for Fast Weights?

In the benchmarks above, we saw two fast-weight precision modes: `float32` and `bfloat16`. In this final section, let's clarify how they differ numerically and which one to pick.

The paper does not specify what floating-point precision should store fast weights at inference time. Inspecting the official repository shows that it clones the original down-projection matrix as fast weights and adds each chunk's $\Delta W$ directly into that tensor. Thus, when the model is loaded in BF16, the fast weights are also BF16, with no separate FP32 master copy.

This creates a numerical hazard: **adding a tiny update directly onto a much larger BF16 matrix loses most of the update to BF16 mantissa rounding (swamping).**

We ran a rounding test on this checkpoint's actual layer-0 down-projection matrix: generating an update matrix of norm `1e-5` (the per-update clipping threshold used in the paper's Qwen3-4B evaluation) and adding it in BF16 directly onto layer 0's matrix changed only `472` out of all `24,903,680` elements, retaining only ~`0.97%` of the intended update magnitude. (Note: this is a numerical demonstration of floating-point rounding, not a model-quality experiment; moreover, this community checkpoint does not enable clipping, so its single-step updates are much larger than `1e-5`.)

To prevent small updates from being swallowed by the base weights, pLLM offers two engineering choices:

- **Default: store deltas separately in FP32 (`dtype=torch.float32`)**: base weights remain untouched in BF16, while each request stores its accumulated `delta` in FP32 and adds `inputs @ delta` to the base output during forward. Even tiny updates accumulate cleanly in FP32 `delta` without rounding away.
- **Official-rounding BF16 mode (`InPlaceTTTWorkload(ttt, dtype=torch.bfloat16)`)**: if you want step-by-step rounding aligned with the official code, pass `dtype=torch.bfloat16` explicitly. After every update, the fast weights are rounded in BF16 just like the official implementation, while the difference between that rounded result and the base weight is still stored in FP32 without further loss.

Both are sound engineering choices; since the paper leaves precision unspecified, pLLM defaults to FP32 mode so small updates are never lost.

### Summary

1. **Core mechanism of In-Place TTT**: add no new layers; treat the MLP down-projection in selected layers as fast weights, updating them every full chunk (e.g., 1024 tokens) using targets computed by a conv1d + projection so subsequent chunks immediately read the updated weights.
2. **Running and inspecting with pLLM**: the complete implementation lives in [`pllm/workloads/inplace_ttt.py`](../../pllm/workloads/inplace_ttt.py). `pllm.explain` reports exact per-request fast-weight VRAM before loading weights; across chunk boundaries during decode, the framework handles fast-weight updates, visibility transitions, and multi-request concurrency automatically.
3. **Connection to earlier examples**: if you read [07 Custom Residual Connections](07_custom_residual.ipynb), you will recognize that the stateful memory sublayer there and In-Place TTT here share the exact same "chunk-updated, read-old-within-chunk" computational structure.